# ComeHear ASR v2: Fine-tuning Whisper untuk Anamnesis Poli Anak (Indonesia + Jawa + istilah medis)

Lanjutan `train-java-indo-voice.ipynb` (v1: whisper-small, WER 34,55% tanpa normalisasi). Perubahan v2:

| Masalah v1 | Perbaikan v2 |
|---|---|
| WER dihitung hanya `lower()`; tanda baca dan angka dihitung salah | Normalisasi teks yang sama dengan backend (`app/scribe/textnorm.py`) |
| Tidak ada pembanding | WER **baseline** (sebelum fine-tune) vs **sesudah**, per domain |
| Tidak ada data percakapan medis | Opsional: kalimat klinis sintetis (TTS) + rekaman role-play tim |
| whisper-small | Pilihan model: `whisper-large-v3-turbo` (default) / `whisper-medium` / `whisper-small` |
| LoRA hanya q,v | LoRA q,k,v,out + SpecAugment |
| Checkpoint terbaik dipilih dengan WER mentah | Dipilih dengan WER ternormalisasi |

Target proposal: WER ≤ 15–20%. Jalankan di Kaggle (GPU T4/P100). Isi bagian **KONFIGURASI** dulu.

In [ ]:
# [1] Instalasi
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q transformers peft datasets evaluate jiwer accelerate soundfile librosa ctranslate2 faster-whisper edge-tts


In [ ]:
# [2] KONFIGURASI
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Kunci ke 1 GPU T4 (hindari DataParallel multi-GPU Kaggle)
QUICK_TRIAL = False           # False: Pelatihan penuh (~25-35 menit). True: Uji cepat 60 langkah (~5-8 menit).
MODEL_ID = 'openai/whisper-large-v3-turbo'   # Bobot whisper akurat & cepat
N_INDONESIAN = 2000          # Sampel Common Voice ID (jika ada)
USE_SYNTHETIC_MEDICAL = False # False jika sudah upload dataset (hemat waktu & kuota). True jika ingin generate TTS di Kaggle.
N_SYNTHETIC = 350
TEAM_RECORDINGS_DIR = '/kaggle/input/comehear-roleplay'  # Opsional: folder dataset custom
LEARNING_RATE = 1e-4
MAX_STEPS = 60 if QUICK_TRIAL else 400   # 400 langkah optimal untuk ~350-500 sampel (~30 menit di GPU T4)
EVAL_STEPS = 30 if QUICK_TRIAL else 100
BATCH = 4 if 'large' in MODEL_ID else 8
GRAD_ACCUM = 4 if 'large' in MODEL_ID else 2
SEED = 42


In [ ]:
# [3] Import + normalisasi teks (salinan app/scribe/textnorm.py agar WER sama dengan backend)
import os
# Batasi ke 1 GPU agar terhindar dari bug DataParallel multi-GPU di Kaggle
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import re, gc, glob, json, random, shutil, sys, unicodedata, inspect, warnings
from dataclasses import dataclass
import numpy as np, pandas as pd, torch, librosa, soundfile as sf
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)
from peft import LoraConfig, get_peft_model, PeftModel
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

_PUNCT = re.compile(r"[^\w\s]"); _SPACE = re.compile(r"\s+")
_ANGKA = {"nol":0,"satu":1,"dua":2,"tiga":3,"empat":4,"lima":5,"enam":6,"tujuh":7,"delapan":8,"sembilan":9,
          "sepuluh":10,"sebelas":11,"seratus":100}
_SINGKATAN = {
    'dok': 'dokter', 'dr': 'dokter', 'bu': 'ibu', 'pak': 'bapak',
    'nggak': 'tidak', 'gak': 'tidak', 'ga': 'tidak', 'enggak': 'tidak', 'ndak': 'tidak', 'ora': 'tidak',
    'udah': 'sudah', 'uda': 'sudah', 'aja': 'saja', 'kalo': 'kalau', 'klo': 'kalau',
    'cuma': 'hanya', 'banget': 'sangat', 'tapi': 'tetapi', 'trus': 'terus', 'abis': 'habis',
    'gimana': 'bagaimana', 'bikin': 'membuat', 'dikasih': 'diberi', 'dikasihkan': 'diberikan',
    'ket': 'sejak', 'wingi': 'kemarin', 'isuk': 'pagi', 'awan': 'siang', 'bengi': 'malam',
    'saiki': 'sekarang', 'dino': 'hari', 'mari': 'selesai', 'sakwise': 'setelah',
    'awakke': 'badannya', 'awakne': 'badannya', 'awake': 'badannya', 'awak': 'badan',
    'sirahne': 'kepalanya', 'sirah': 'kepala', 'anake': 'anaknya', 'bocah': 'anak', 'arek': 'anak',
    'sumer': 'demam', 'nggreges': 'meriang', 'ngelu': 'pusing', 'lemes': 'lemas',
    'mencret': 'diare', 'senep': 'mules', 'mutah': 'muntah', 'watuk': 'batuk',
    'sambat': 'mengeluh', 'gatel': 'gatal', 'ngombe': 'minum', 'mangan': 'makan',
    'turu': 'tidur', 'pipis': 'kencing',
    'siji': 'satu', 'loro': 'dua', 'rong': 'dua', 'telu': 'tiga', 'telung': 'tiga',
    'papat': 'empat', 'patang': 'empat', 'limo': 'lima', 'enem': 'enam',
    'pitu': 'tujuh', 'wolu': 'delapan', 'songo': 'sembilan',
}

def _words_to_digits(words):
    out, i = [], 0
    while i < len(words):
        w = words[i]
        if w in _ANGKA:
            n = _ANGKA[w]
            if i + 1 < len(words) and words[i+1] == "puluh":
                n, i = n * 10, i + 1
                if i + 1 < len(words) and words[i+1] in _ANGKA and _ANGKA[words[i+1]] < 10:
                    n, i = n + _ANGKA[words[i+1]], i + 1
            elif i + 1 < len(words) and words[i+1] == "belas":
                n, i = n + 10, i + 1
            out.append(str(n))
        else:
            out.append(w)
        i += 1
    return out

def normalize(t):
    t = unicodedata.normalize("NFKC", t).lower()
    t = re.sub(r"(\d)[,\.](\d)", r"\1 koma \2", t).replace("-", " ")
    words = _SPACE.sub(" ", _PUNCT.sub(" ", t)).strip().split()
    return " ".join(_words_to_digits([_SINGKATAN.get(w, w) for w in words]))

assert normalize("Suhu 38,6 derajat, Dok!") == "suhu 38 koma 6 derajat dokter"

In [ ]:
# [4] Data: Common Voice ID + Jawa-Lumajang + Rekaman Role-Play + Data Klinis Sintetis
KAGGLE = '/kaggle/input'
def valid(p, min_bytes=2048):
    try: return os.path.exists(p) and os.path.getsize(p) > min_bytes
    except OSError: return False

def load_audio_csv(meta_path, domain_label='roleplay_klinis'):
    '''Membaca CSV dataset audio dengan fleksibel terhadap nama kolom dan struktur folder.'''
    if not os.path.exists(meta_path):
        return None
    try: df = pd.read_csv(meta_path)
    except Exception:
        return None
    root_dir = os.path.dirname(meta_path)
    audio_col = next((c for c in ['audio_path', 'audio', 'file', 'filename', 'path', 'wav'] if c in df.columns), None)
    text_col = next((c for c in ['sentence', 'text', 'transcription', 'transcript'] if c in df.columns), None)
    if not audio_col or not text_col:
        return None

    def resolve_path(p):
        p_str = str(p).strip()
        if os.path.exists(p_str) and valid(p_str): return p_str
        fname = os.path.basename(p_str.replace('\\', '/'))
        for cand in [os.path.join(root_dir, 'audio', fname), os.path.join(root_dir, fname)]:
            if os.path.exists(cand) and valid(cand): return cand
        matches = glob.glob(os.path.join(root_dir, '**', fname), recursive=True)
        if matches and valid(matches[0]): return matches[0]
        return None

    df['resolved_path'] = df[audio_col].map(resolve_path)
    df = df[df['resolved_path'].notna() & df[text_col].notna()]
    if len(df) == 0: return None
    return pd.DataFrame({'audio_path': df['resolved_path'], 'sentence': df[text_col].astype(str).str.strip(), 'domain': domain_label})

frames = []

# 1. Dataset Common Voice ID (jika ada)
cv = glob.glob(f'{KAGGLE}/**/train.tsv', recursive=True) or glob.glob(f'{KAGGLE}/**/validated.tsv', recursive=True)
if cv:
    root = os.path.dirname(cv[0])
    df = pd.read_csv(cv[0], sep='\t', low_memory=False)
    df['audio_path'] = df['path'].map(lambda x: os.path.join(root, 'clips', str(x)))
    df = df[df['sentence'].notna() & df['audio_path'].map(valid)]
    df = df.sample(n=min(100 if QUICK_TRIAL else N_INDONESIAN, len(df)), random_state=SEED)
    frames.append(pd.DataFrame({'audio_path': df['audio_path'], 'sentence': df['sentence'].str.strip(), 'domain': 'indonesia'}))
    print(f'Common Voice ID dimuat: {len(df)} sampel')

# 2. Dataset Dialek Jawa Lumajang (jika ada)
jw = glob.glob(f'{KAGGLE}/**/Mapping.tsv', recursive=True)
if jw:
    root = os.path.dirname(jw[0])
    df = pd.read_csv(jw[0], sep='\t')
    df['audio_path'] = df.apply(lambda r: os.path.join(root, str(r['speaker_number']), str(r['audio_filename'])), axis=1)
    df = df[df['post-edited transcription'].notna() & df['audio_path'].map(lambda p: valid(p, 1024))]
    if QUICK_TRIAL: df = df.sample(n=min(50, len(df)), random_state=SEED)
    frames.append(pd.DataFrame({'audio_path': df['audio_path'], 'sentence': df['post-edited transcription'].str.strip(), 'domain': 'jawa'}))
    print(f'Jawa Lumajang dimuat: {len(df)} sampel')

# 3. Cari dan muat SEMUA dataset klinis / roleplay dari input Kaggle
candidate_metas = glob.glob(f'{KAGGLE}/**/metadata.csv', recursive=True) + \
                  glob.glob('../../datasets/synthetic_clinical/metadata.csv') + \
                  glob.glob('../datasets/synthetic_clinical/metadata.csv') + \
                  glob.glob('./synthetic_clinical/metadata.csv')

loaded_metas = set()
for m in candidate_metas:
    abs_m = os.path.abspath(m)
    if abs_m in loaded_metas: continue
    res = load_audio_csv(m, domain_label='roleplay_klinis')
    if res is not None:
        loaded_metas.add(abs_m)
        if QUICK_TRIAL and len(res) > 50: res = res.sample(n=50, random_state=SEED)
        frames.append(res)
        print(f'Dataset klinis role-play (gold data) dimuat: {len(res)} sampel dari {m}')
        break


### Data klinis sintetis (opsional)
Common Voice dan dialek Lumajang tidak berisi percakapan dokter. Sel berikut membuat kalimat anamnesis poli anak
dari templat (istilah klinis, angka suhu/berat, dosis, campuran Jawa) lalu menyuarakannya dengan
`facebook/mms-tts-ind` plus derau ringan dan variasi kecepatan. Porsinya sengaja kecil (±20%) supaya model
belajar **kosakata** tanpa terlalu menyesuaikan diri ke suara TTS.

Catatan lisensi: MMS-TTS berlisensi CC-BY-NC 4.0 (non-komersial). Aman untuk prototipe lomba; untuk produk
SaaS ganti dengan rekaman role-play tim (`TEAM_RECORDINGS_DIR`) atau TTS berlisensi komersial.

In [ ]:
# [5] Kalimat klinis sintetis + Multi-speaker TTS (HANYA JALAN JIKA BELUM ADA DATASET)
import threading, asyncio

KELUHAN = [
    'demam', 'panas', 'batuk berdahak', 'batuk kering', 'pilek', 'mencret', 'diare', 'muntah', 'mual',
    'sesak napas', 'nyeri perut', 'sakit kepala', 'ruam merah', 'biduran', 'gatal-gatal', 'kejang',
    'mimisan', 'nafsu makan turun', 'rewel', 'lemas', 'sumer', 'ngelu', 'batuk grok-grok',
    'BAB nya cair', 'feses berlendir', 'perut mules', 'ada lendir', 'ada darah pada BAB'
]
DURASI = [
    'sejak kemarin', 'sejak tadi pagi', 'sudah dua hari', 'sudah tiga hari', 'sudah seminggu',
    'ket wingi', 'ket isuk', 'naik turun sejak lusa'
]
OBAT = [
    'parasetamol sirup', 'ibuprofen sirup', 'amoksisilin', 'oralit', 'zink', 'salbutamol', 'cetirizine',
    'tempra', 'sanmol', 'proris', 'obat anti diare', 'puyer batuk'
]
PEMICU = [
    'jajan seblak pedas', 'makan lauk sambal pedas', 'minum es sirup', 'jajan sembarangan',
    'tumbas seblak level tiga'
]
ANGKA = [
    'tiga puluh delapan koma lima', 'tiga puluh sembilan', 'tiga puluh tujuh koma delapan',
    'empat puluh', 'tiga puluh delapan koma sembilan'
]
POLA = [
    'Monggo pinarak, silakan duduk dulu dok. Anak saya ini BAB nya cair terus sejak kemarin.',
    'Duduk dulu ibu. BAB nya cair berapa kali sehari? Ada lendir atau darah pada BAB nya?',
    'Mboten wonten darah dok, tapi fesesnya encer dan ada lendir sedikit.',
    'Lauknya diperhatikan ya bu, jangan jajan seblak atau makanan pedas dulu.',
    'Adiknya masih mau makan tidak? Tetep mangan dok, nafsu makannya alhamdulillah masih ada.',
    'Tolong kasih makanan yang mudah dicerna dulu ya bu, seperti bubur halus dan sup hangat.',
    'Untuk obat anti diare jangan sembarangan diminumkan nggih bu, cukup oralit dan zink.',
    'Obat sirupnya ini bisa dituku langsung di apotek terdekat ya bu.',
    'Perhatikan ya bu, kalau ada darah pada BAB segera bawa ke rumah sakit.',
    'Anak saya {k} {d}, Dok, sehabis {p}.',
    'Ibu, adiknya {k} {d}? Sudah dikasih {o} di rumah?',
    'Suhunya tadi {a} derajat, badannya sumer terus ngelu sirahe.',
    'Selain {k}, keluhan mual muntah mboten enten dok.',
    'Larene tetep mangan sekedik-sekedik nggih dok senajan wetenge mules?',
    'Saya resepkan {o} lima mililiter tiap empat sampai enam jam kalau demam.',
    'Berat badannya {b} kilo, minumnya berkurang tapi buang air kecil masih lancar.',
    'Dok, niki wau larene nyuwun seblak, lajeng wetenge mules terus BAB mencret lendir.',
    'Nggih bu, monggo duduk dulu. Obat anti diare mboten pareng dipun paringaken sembarangan.'
]
def synth_sentence():
    k = random.choice(KELUHAN)
    p = random.choice(PEMICU)
    b = random.choice(['sembilan', 'dua belas', 'lima belas', 'delapan belas', 'dua puluh dua'])
    return random.choice(POLA).format(k=k, d=random.choice(DURASI), o=random.choice(OBAT), p=p, a=random.choice(ANGKA), b=b)

has_clinical_audio = any(f.get('domain', '').iloc[0] == 'roleplay_klinis' for f in frames if len(f) > 0)

if has_clinical_audio:
    total_samples = sum(len(f) for f in frames)
    print(f'[INFO] Dataset audio klinis sudah dimuat di Sel [4] ({total_samples} total sampel).')
    print('[INFO] Generator TTS dilewati agar tidak membuang waktu & kuota GPU!')
elif USE_SYNTHETIC_MEDICAL:
    out_dir = '/kaggle/working/synthetic'; os.makedirs(out_dir, exist_ok=True)
    rows = []
    print('[INFO] Membangun data klinis sintetis via Edge-TTS...')
    try:
        import edge_tts, av
        async def gen_edge():
            voices = ['id-ID-ArdiNeural', 'id-ID-GadisNeural', 'jv-ID-DimasNeural', 'jv-ID-SitiNeural']
            count = 40 if QUICK_TRIAL else N_SYNTHETIC
            for n in range(count):
                text = synth_sentence()
                comm = edge_tts.Communicate(text, random.choice(voices))
                buf = bytearray()
                async for chunk in comm.stream():
                    if chunk['type'] == 'audio': buf.extend(chunk['data'])
                c = av.open(io.BytesIO(buf))
                r = av.AudioResampler(format='s16', layout='mono', rate=16000)
                f_list = [rf.to_ndarray() for fr in c.decode(audio=0) for rf in r.resample(fr)]
                wav = np.concatenate(f_list, axis=1).squeeze()
                p_wav = f'{out_dir}/syn_{n:05d}.wav'
                sf.write(p_wav, wav, 16000)
                rows.append({'audio_path': p_wav, 'sentence': text, 'domain': 'roleplay_klinis'})
        def run_thread():
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(gen_edge())
            loop.close()
        t = threading.Thread(target=run_thread)
        t.start(); t.join()
    except Exception as e:
        print(f'[WARN] Edge-TTS gagal ({e}), fallback ke mms-tts cepat...')
        from transformers import VitsModel, AutoTokenizer
        tts_tok = AutoTokenizer.from_pretrained('facebook/mms-tts-ind')
        tts = VitsModel.from_pretrained('facebook/mms-tts-ind').to(device).eval()
        count = 40 if QUICK_TRIAL else min(100, N_SYNTHETIC)
        for n in range(count):
            text = synth_sentence()
            with torch.no_grad():
                wav = tts(**tts_tok(text, return_tensors='pt').to(device)).waveform[0].cpu().numpy()
            p_wav = f'{out_dir}/syn_{n:05d}.wav'
            sf.write(p_wav, wav.astype(np.float32), tts.config.sampling_rate)
            rows.append({'audio_path': p_wav, 'sentence': text, 'domain': 'roleplay_klinis'})
        del tts; gc.collect(); torch.cuda.empty_cache()
    if rows:
        frames.append(pd.DataFrame(rows))

if not frames:
    raise RuntimeError('Tidak ada data audio yang ditemukan! Pastikan Anda sudah menambahkan dataset di tab Input Kaggle (misal synthetic-clinical-jawa-id).')
data = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(data['domain'].value_counts())


In [ ]:
# [6] Split per domain: train / val (pilih checkpoint) / test (laporan akhir, tidak pernah dilihat saat latih)
strat = data["domain"] if len(data["domain"].unique()) > 1 else None
train_df, rest = train_test_split(data, test_size=0.2, random_state=SEED, stratify=strat)
strat_rest = rest["domain"] if len(rest["domain"].unique()) > 1 else None
val_df, test_df = train_test_split(rest, test_size=0.5, random_state=SEED, stratify=strat_rest)
print(len(train_df), len(val_df), len(test_df))

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="indonesian", task="transcribe")

def prepare(batch):
    try:
        audio, _ = librosa.load(batch["audio_path"], sr=16000)
    except Exception:
        audio = np.zeros(16000, dtype=np.float32)
    batch["input_features"] = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids[:440]
    return batch

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df[["audio_path", "sentence"]].reset_index(drop=True)),
    "val": Dataset.from_pandas(val_df[["audio_path", "sentence"]].reset_index(drop=True)),
}).map(prepare, remove_columns=["audio_path", "sentence"], writer_batch_size=100)
gc.collect()

@dataclass
class Collator:
    processor: object
    def __call__(self, features):
        batch = self.processor.feature_extractor.pad([{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        labels = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        ids = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (ids[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            ids = ids[:, 1:]
        batch["labels"] = ids
        return batch

In [ ]:
# [7] Fungsi evaluasi WER/CER ternormalisasi per domain (dipakai untuk baseline dan hasil akhir)
import jiwer

def transcribe_df(model, df, batch=8):
    model.eval(); preds = []
    for i in range(0, len(df), batch):
        rows = df.iloc[i:i + batch]
        audios = [librosa.load(p, sr=16000)[0] for p in rows["audio_path"]]
        feats = processor.feature_extractor(audios, sampling_rate=16000, return_tensors="pt").input_features
        with torch.no_grad(), torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            ids = model.generate(input_features=feats.to(device, dtype=model.dtype), language="indonesian",
                                 task="transcribe", max_new_tokens=220, num_beams=1)
        preds += processor.batch_decode(ids, skip_special_tokens=True)
    return preds

def report(model, df, title):
    preds = transcribe_df(model, df)
    out = df.assign(pred=preds)
    rows = []
    for dom, g in list(out.groupby("domain")) + [("SEMUA", out)]:
        refs = [normalize(s) for s in g["sentence"]]; hyps = [normalize(p) for p in g["pred"]]
        keep = [i for i, r in enumerate(refs) if r]
        rows.append({"domain": dom, "n": len(keep),
                     "WER%": round(100 * jiwer.wer([refs[i] for i in keep], [hyps[i] for i in keep]), 2),
                     "CER%": round(100 * jiwer.cer([refs[i] for i in keep], [hyps[i] for i in keep]), 2)})
    res = pd.DataFrame(rows); print(f"\n== {title} =="); print(res.to_string(index=False))
    return res, out

base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
baseline, _ = report(base, test_df if not QUICK_TRIAL else test_df.head(40), f"BASELINE {MODEL_ID} (tanpa fine-tune)")
del base; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# [8] LoRA + pelatihan (checkpoint terbaik dipilih dengan WER ternormalisasi)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.use_cache = False
model.generation_config.language = "indonesian"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None
model.config.begin_suppress_tokens = None
try:
    import peft.import_utils; peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass
model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
                                         target_modules=["q_proj", "k_proj", "v_proj", "out_proj"]))
model.print_trainable_parameters()

def compute_metrics(pred):
    ids, labels = pred.predictions, pred.label_ids
    labels[labels == -100] = processor.tokenizer.pad_token_id
    p = [normalize(x) for x in processor.tokenizer.batch_decode(ids, skip_special_tokens=True)]
    r = [normalize(x) for x in processor.tokenizer.batch_decode(labels, skip_special_tokens=True)]
    keep = [i for i, x in enumerate(r) if x]
    return {"wer": 100 * jiwer.wer([r[i] for i in keep], [p[i] for i in keep]),
            "cer": 100 * jiwer.cer([r[i] for i in keep], [p[i] for i in keep])}

args = Seq2SeqTrainingArguments(
    output_dir="./asr-v2", per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, warmup_steps=max(10, MAX_STEPS // 10), max_steps=MAX_STEPS, max_grad_norm=1.0,
    gradient_checkpointing=True, fp16=device == "cuda", eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS, logging_steps=max(10, EVAL_STEPS // 5), per_device_eval_batch_size=BATCH,
    predict_with_generate=True, generation_max_length=225, report_to=[], load_best_model_at_end=True,
    metric_for_best_model="wer", greater_is_better=False, save_total_limit=2, remove_unused_columns=False,
    label_names=["labels"], seed=SEED)
kw = dict(args=args, model=model, train_dataset=ds["train"], eval_dataset=ds["val"],
          data_collator=Collator(processor), compute_metrics=compute_metrics)
params = inspect.signature(Seq2SeqTrainer.__init__).parameters
kw["processing_class" if "processing_class" in params else "tokenizer"] = processor.feature_extractor
# Cegah DataParallel multi-GPU & amankan autocast FP16 saat evaluasi generate
class SingleGPUTrainer(Seq2SeqTrainer):
    def _wrap_model(self, model, training=True, dataloader=None):
        return model

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None, **gen_kwargs):
        if "input_features" in inputs and torch.is_tensor(inputs["input_features"]):
            if inputs["input_features"].dtype == torch.float32 and self.args.fp16:
                inputs["input_features"] = inputs["input_features"].to(torch.float16)
        with torch.autocast(self.args.device.type, dtype=torch.float16, enabled=self.args.fp16):
            return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys=ignore_keys, **gen_kwargs)

trainer = SingleGPUTrainer(**kw)
trainer.train()
trainer.save_model("./asr-v2-lora")
model.save_pretrained("./asr-v2-lora")
processor.save_pretrained("./asr-v2-lora")
print("[OK] Pelatihan selesai & model LoRA berhasil disimpan di ./asr-v2-lora")


In [ ]:
# [9] Gabung LoRA, evaluasi di TEST set, bandingkan dengan baseline
import os, glob, torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

MODEL_ID = globals().get('MODEL_ID', 'openai/whisper-large-v3-turbo')
device = globals().get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
if 'processor' not in globals():
    processor = WhisperProcessor.from_pretrained(MODEL_ID, language='indonesian', task='transcribe')

# Cek ketersediaan adapter LoRA
lora_dir = './asr-v2-lora'
if not os.path.exists(os.path.join(lora_dir, 'adapter_config.json')):
    checkpoints = sorted(glob.glob('./asr-v2/checkpoint-*'), key=os.path.getmtime)
    if checkpoints and os.path.exists(os.path.join(checkpoints[-1], 'adapter_config.json')):
        lora_dir = checkpoints[-1]
        print(f'[INFO] Menggunakan adapter dari checkpoint: {lora_dir}')
    else:
        raise FileNotFoundError(
            'Folder ./asr-v2-lora belum ditemukan! ' + 
            'Pastikan Sel [8] sudah dijalankan dan selesai sebelum menjalankan Sel [9].'
        )

print(f'[INFO] Memuat model dasar: {MODEL_ID}...')
base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
print(f'[INFO] Menggabungkan adapter LoRA dari {lora_dir}...')
merged = PeftModel.from_pretrained(base, lora_dir).merge_and_unload()
merged.generation_config.forced_decoder_ids = None
out_merged = './asr-v2-merged'
merged.save_pretrained(out_merged)
processor.save_pretrained(out_merged)
try:
    processor.feature_extractor.save_pretrained(out_merged)
    processor.tokenizer.save_pretrained(out_merged)
except Exception:
    pass
print(f'[OK] Model berhasil digabung dan disimpan di {out_merged}')

if 'report' in globals() and 'test_df' in globals():
    merged = merged.to(device).half() if device == 'cuda' else merged.to(device)
    final, preds = report(merged, test_df if not globals().get('QUICK_TRIAL', True) else test_df.head(40), 'SESUDAH FINE-TUNE (test set)')
    if 'baseline' in globals():
        cmp = baseline.merge(final, on=['domain', 'n'], suffixes=(' baseline', ' v2'))
        print('\n== PERBANDINGAN =='); print(cmp.to_string(index=False))
        cmp.to_csv('wer_report.csv', index=False)
    preds.to_csv('test_predictions.csv', index=False)
else:
    print('[INFO] Model berhasil digabung! Lanjut ke Sel [10] untuk ekspor CTranslate2.')


In [ ]:
# [10] Ekspor CTranslate2 untuk backend (ASR_MODEL=<folder> di .env), dikompres jadi zip
import os, sys, shutil
from ctranslate2.converters import TransformersConverter
from huggingface_hub import hf_hub_download

MODEL_ID = globals().get('MODEL_ID', 'openai/whisper-large-v3-turbo')
CT2 = './comehear-asr-v2-ct2'
merged_dir = './asr-v2-merged'

print('[INFO] Mengonversi bobot PyTorch ke format CTranslate2 float16...')
converter = TransformersConverter(merged_dir, load_as_float16=True)
converter.convert(CT2, quantization='float16', force=True)

print('[INFO] Memverifikasi kelengkapan file tokenizer & preprocessor...')
# File pendukung wajib untuk runtime faster-whisper
required_files = ['tokenizer.json', 'preprocessor_config.json', 'tokenizer_config.json', 'generation_config.json']
for fname in required_files:
    src = os.path.join(merged_dir, fname)
    dst = os.path.join(CT2, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  [OK] {fname} disalin dari {merged_dir}')
    else:
        # Jika belum ada di merged_dir, unduh otomatis dari model Whisper resmi di HuggingFace
        try:
            hf_path = hf_hub_download(repo_id=MODEL_ID, filename=fname)
            shutil.copy2(hf_path, dst)
            print(f'  [OK] {fname} diunduh otomatis dari {MODEL_ID}')
        except Exception as e:
            print(f'  [INFO] {fname} opsional: {e}')

zip_file = shutil.make_archive('comehear-asr-v2-ct2', 'zip', CT2)
size_mb = os.path.getsize(zip_file) / (1024 * 1024)
print(f'\n[SUKSES] Arsip model {os.path.basename(zip_file)} ({size_mb:.1f} MB) berhasil dibuat!')
print('Unduh file comehear-asr-v2-ct2.zip dari tab Output di panel kanan Kaggle.')


In [ ]:
# [11] Uji cepat faster-whisper (sama persis dengan backend: CPU int8 bila tanpa GPU)
import os, glob, librosa
from faster_whisper import WhisperModel

CT2 = './comehear-asr-v2-ct2'

# Inisialisasi model: Gunakan CPU int8 (persis backend) atau fallback jika libcublas GPU tidak tersedia
try:
    fw = WhisperModel(CT2, device='cpu', compute_type='int8')
    print('[OK] faster-whisper berhasil dimuat dalam mode CPU int8 (sama persis dengan backend lokal)!')
except Exception as e:
    print(f'[WARN] CPU int8 gagal ({e}), mencoba default...')
    fw = WhisperModel(CT2, device='auto', compute_type='default')

if 'test_df' in globals():
    print('[INFO] Menguji 5 sampel dari test set...')
    for _, r in test_df.head(5).iterrows():
        wav, _ = librosa.load(r['audio_path'], sr=16000)
        segs, _ = fw.transcribe(wav, language='id', beam_size=5, condition_on_previous_text=False)
        hyp = ' '.join(s.text for s in segs).strip()
        print(f"[{r['domain']}] REF: {r['sentence']}\\n           HYP: {hyp}\\n")
else:
    sample_audios = glob.glob('/kaggle/input/**/*.wav', recursive=True)
    if sample_audios:
        print(f'[INFO] Menguji coba inferensi pada {os.path.basename(sample_audios[0])}...')
        wav, _ = librosa.load(sample_audios[0], sr=16000)
        segs, _ = fw.transcribe(wav, language='id', beam_size=5)
        print('Hasil transkripsi:', ' '.join(s.text for s in segs).strip())
    else:
        print('[OK] faster-whisper berhasil diinisialisasi dan siap digunakan!')
